In [1]:
# Model Evaluation Script for All Models
import os
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2, 
    FasterRCNN_ResNet50_FPN_V2_Weights,
    ssd300_vgg16, 
    SSD300_VGG16_Weights,
    ssdlite320_mobilenet_v3_large, 
    SSDLite320_MobileNet_V3_Large_Weights,
    _utils
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssd import SSDClassificationHead
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from sklearn.model_selection import KFold
from tqdm import tqdm
import numpy as np
import pandas as pd
from PIL import Image
from ultralytics import YOLO

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

NUM_FOLDS = 4
NUM_CLASSES = 4  # background + 3 classes
CLASS_NAMES = ["pothole", "crack", "manhole"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset paths
IMAGE_DIR = "datasets/dataset_scientific_reports/data/images"
LABEL_DIR = "datasets/dataset_scientific_reports/data/labels"

# Model configurations
MODEL_CONFIGS = {
    'faster_rcnn': {
        'input_size': 300,
        'batch_size': 8,
        'conf_threshold': 0.25,
    },
    'ssd300': {
        'input_size': 300,
        'batch_size': 16,
        'conf_threshold': 0.25,
    },
    'ssdlite320': {
        'input_size': 320,
        'batch_size': 16,
        'conf_threshold': 0.25,
    },
    'yolov8n': {
        'input_size': 640,
        'batch_size': 16,
        'conf_threshold': 0.25,
    },
    'yolov11n': {
        'input_size': 640,
        'batch_size': 16,
        'conf_threshold': 0.25,
    },
    'yolov11s': {
        'input_size': 640,
        'batch_size': 16,
        'conf_threshold': 0.25,
    },
}

# Reproducibility
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Device: {DEVICE}")
print(f"Models to evaluate: {list(MODEL_CONFIGS.keys())}")

Device: cpu
Models to evaluate: ['faster_rcnn', 'ssd300', 'ssdlite320', 'yolov11n', 'yolov11s']


In [3]:
# ============================================================================
# DATASET CLASS
# ============================================================================

class PotholeDataset(Dataset):
    def __init__(self, image_dir, label_dir, image_files, input_size, model_type='faster_rcnn'):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = image_files
        self.input_size = input_size
        self.model_type = model_type
        self.transform = transforms.ToTensor()

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        original_w, original_h = image.size

        label_name = img_name.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(self.label_dir, label_name)

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 9:
                        class_id = int(parts[0])
                        coords = list(map(float, parts[1:]))

                        xs = coords[0::2]
                        ys = coords[1::2]

                        x_min = min(xs) * original_w
                        y_min = min(ys) * original_h
                        x_max = max(xs) * original_w
                        y_max = max(ys) * original_h

                        w = x_max - x_min
                        h = y_max - y_min

                        # Filter small boxes
                        if w >= 5.0 and h >= 5.0:
                            boxes.append([x_min, y_min, x_max, y_max])
                            labels.append(class_id + 1)

        # Resize based on model type
        if self.model_type == 'faster_rcnn':
            # Maintain aspect ratio for Faster R-CNN
            scale = self.input_size / max(original_w, original_h)
            new_w = int(original_w * scale)
            new_h = int(original_h * scale)
            image = image.resize((new_w, new_h))
            scale_x = new_w / original_w
            scale_y = new_h / original_h
        else:
            # Fixed size for SSD models
            image = image.resize((self.input_size, self.input_size))
            scale_x = self.input_size / original_w
            scale_y = self.input_size / original_h

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            boxes[:, [0, 2]] *= scale_x
            boxes[:, [1, 3]] *= scale_y

        # Calculate area
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) > 0 else torch.zeros((0,), dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64)
        }

        image = self.transform(image)

        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

In [4]:
# ============================================================================
# MODEL CREATION FUNCTIONS
# ============================================================================

def create_faster_rcnn(num_classes=4, trainable_backbone_layers=3):
    """Create Faster R-CNN with ResNet50-FPN backbone (V2)"""
    model = fasterrcnn_resnet50_fpn_v2(
        weights=FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1,
        trainable_backbone_layers=trainable_backbone_layers
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def create_ssd300(num_classes=4, size=300):
    """Create SSD300 with VGG16 backbone"""
    model = ssd300_vgg16(weights=SSD300_VGG16_Weights.COCO_V1)
    
    in_channels = _utils.retrieve_out_channels(model.backbone, (size, size))
    num_anchors = model.anchor_generator.num_anchors_per_location()
    
    model.head.classification_head = SSDClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes
    )
    
    model.transform.min_size = (size,)
    model.transform.max_size = size
    
    return model

def create_ssdlite320(num_classes=4, size=320):
    """Create SSD320 with MobileNetV3 backbone"""
    model = ssdlite320_mobilenet_v3_large(weights=SSDLite320_MobileNet_V3_Large_Weights.COCO_V1)
    
    in_channels = _utils.retrieve_out_channels(model.backbone, (size, size))
    num_anchors = model.anchor_generator.num_anchors_per_location()
    
    model.head.classification_head = SSDClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes
    )
    
    model.transform.min_size = (size,)
    model.transform.max_size = size
    
    return model

In [5]:
# ============================================================================
# EVALUATION FUNCTIONS
# ============================================================================

@torch.no_grad()
def evaluate_pytorch_model(model, dataloader, conf_threshold, device):
    """Evaluate PyTorch detection model"""
    model.eval()
    metric = MeanAveragePrecision(class_metrics=True)
    
    for images, targets in tqdm(dataloader, desc="Evaluating", leave=False):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        outputs = model(images)
        
        # Filter by confidence
        preds = []
        for output in outputs:
            keep = output['scores'] >= conf_threshold
            pred = {
                'boxes': output['boxes'][keep],
                'scores': output['scores'][keep],
                'labels': output['labels'][keep],
            }
            preds.append(pred)
        
        # Move to CPU for metric computation
        preds = [{k: v.cpu() for k, v in p.items()} for p in preds]
        targets = [{k: v.cpu() for k, v in t.items()} for t in targets]
        
        metric.update(preds, targets)
    
    return metric.compute()

@torch.no_grad()
def evaluate_yolo_model(model_path, dataloader, conf_threshold):
    """Evaluate YOLO model"""
    model = YOLO(model_path)
    metric = MeanAveragePrecision(class_metrics=True)
    
    for images, targets in tqdm(dataloader, desc="Evaluating", leave=False):
        # YOLO expects PIL images or numpy arrays
        preds = []
        
        for img_tensor in images:
            # Convert tensor to numpy
            img_np = (img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
            
            # Run inference
            results = model.predict(img_np, conf=conf_threshold, verbose=False)[0]
            
            # Convert to torchmetrics format
            if len(results.boxes) > 0:
                pred = {
                    'boxes': torch.tensor(results.boxes.xyxy.cpu().numpy(), dtype=torch.float32),
                    'scores': torch.tensor(results.boxes.conf.cpu().numpy(), dtype=torch.float32),
                    'labels': torch.tensor(results.boxes.cls.cpu().numpy(), dtype=torch.int64) + 1,  # +1 for background class
                }
            else:
                pred = {
                    'boxes': torch.zeros((0, 4), dtype=torch.float32),
                    'scores': torch.zeros((0,), dtype=torch.float32),
                    'labels': torch.zeros((0,), dtype=torch.int64),
                }
            preds.append(pred)
        
        # Targets already on CPU
        metric.update(preds, targets)
    
    return metric.compute()

In [6]:
# ============================================================================
# PREPARE FOLDS
# ============================================================================

print("\nPreparing folds...")
image_files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
random.shuffle(image_files)
print(f"Total images: {len(image_files)}")

kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=0)
fold_splits = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(image_files)):
    train_files = [image_files[i] for i in train_idx]
    val_files = [image_files[i] for i in val_idx]
    fold_splits.append({'train': train_files, 'val': val_files})
    print(f"Fold {fold_idx}: {len(train_files)} train, {len(val_files)} val")

print("\nFolds prepared.")


Preparing folds...
Total images: 2009
Fold 0: 1506 train, 503 val
Fold 1: 1507 train, 502 val
Fold 2: 1507 train, 502 val
Fold 3: 1507 train, 502 val

Folds prepared.


In [8]:
# ============================================================================
# MAIN EVALUATION LOOP
# ============================================================================

all_results = []

for model_name, config in MODEL_CONFIGS.items():
    print(f"\n{'='*80}")
    print(f"Evaluating {model_name.upper()}")
    print(f"{'='*80}")
    
    for fold_idx in range(NUM_FOLDS):
        print(f"\nFold {fold_idx}/{NUM_FOLDS-1}")
        
        # Load checkpoint path
        if model_name.startswith('yolo'):
            checkpoint_path = f"models/{model_name}/fold_{fold_idx}.pt"
        else:
            checkpoint_path = f"models/{model_name}/fold_{fold_idx}.pth"

        print(checkpoint_path)
        
        if not os.path.exists(checkpoint_path):
            print(f"⚠️  Checkpoint not found: {checkpoint_path}")
            continue
        
        # Create validation dataset
        val_files = fold_splits[fold_idx]['val']
        model_type = 'faster_rcnn' if model_name == 'faster_rcnn' else 'ssd'
        
        val_dataset = PotholeDataset(
            IMAGE_DIR, 
            LABEL_DIR, 
            val_files, 
            config['input_size'],
            model_type=model_type
        )
        
        val_loader = DataLoader(
            val_dataset,
            batch_size=config['batch_size'],
            shuffle=False,
            collate_fn=collate_fn,
            num_workers=2
        )
        
        print(f"Validation samples: {len(val_dataset)}")
        
        # Evaluate based on model type
        if model_name.startswith('yolo'):
            # YOLO models
            metrics = evaluate_yolo_model(
                checkpoint_path,
                val_loader,
                config['conf_threshold']
            )
        else:
            # PyTorch detection models
            if model_name == 'faster_rcnn':
                model = create_faster_rcnn(NUM_CLASSES)
            elif model_name == 'ssd300':
                model = create_ssd300(NUM_CLASSES, config['input_size'])
            elif model_name == 'ssdlite320':
                model = create_ssdlite320(NUM_CLASSES, config['input_size'])
            
            # Load checkpoint
            checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
            if 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            else:
                model.load_state_dict(checkpoint)
            
            model.to(DEVICE)
            
            # Evaluate
            metrics = evaluate_pytorch_model(
                model,
                val_loader,
                config['conf_threshold'],
                DEVICE
            )
            
            # Free memory
            del model
            torch.cuda.empty_cache()
        
        # Extract metrics
        result = {
            'model': model_name,
            'fold': fold_idx,
            'map': metrics['map'].item(),
            'map_50': metrics['map_50'].item(),
            'map_75': metrics['map_75'].item(),
            'map_small': metrics['map_small'].item(),
            'map_medium': metrics['map_medium'].item(),
            'map_large': metrics['map_large'].item(),
            'mar_1': metrics['mar_1'].item(),
            'mar_10': metrics['mar_10'].item(),
            'mar_100': metrics['mar_100'].item(),
            'mar_small': metrics['mar_small'].item(),
            'mar_medium': metrics['mar_medium'].item(),
            'mar_large': metrics['mar_large'].item(),
        }
        
        # Per-class metrics
        if 'map_per_class' in metrics and metrics['map_per_class'].dim() > 0:
            map_per_class = metrics['map_per_class'].cpu().numpy()
            for i, class_name in enumerate(CLASS_NAMES):
                if i < len(map_per_class):
                    result[f'map_{class_name}'] = float(map_per_class[i])
        
        if 'mar_100_per_class' in metrics and metrics['mar_100_per_class'].dim() > 0:
            mar_per_class = metrics['mar_100_per_class'].cpu().numpy()
            for i, class_name in enumerate(CLASS_NAMES):
                if i < len(mar_per_class):
                    result[f'mar_{class_name}'] = float(mar_per_class[i])
        
        all_results.append(result)
        
        # Print key metrics
        print(f"  mAP: {result['map']:.4f}")
        print(f"  mAP@50: {result['map_50']:.4f}")
        print(f"  mAP@75: {result['map_75']:.4f}")


Evaluating FASTER_RCNN

Fold 0/3
models/faster_rcnn/fold_0.pth
Validation samples: 503


/tmp/ipykernel_58320/926419041.py:67: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=DEVICE)


KeyboardInterrupt: 

In [ ]:
# ============================================================================
# SAVE RESULTS
# ============================================================================

print(f"\n{'='*80}")
print("SAVING RESULTS")
print(f"{'='*80}")

# Create DataFrame
df = pd.DataFrame(all_results)

# Save detailed results
output_file = 'model_evaluation_results.csv'
df.to_csv(output_file, index=False)
print(f"\n✓ Detailed results saved to: {output_file}")

# Compute and save summary statistics
summary_results = []

for model_name in MODEL_CONFIGS.keys():
    model_df = df[df['model'] == model_name]
    
    if len(model_df) > 0:
        summary = {
            'model': model_name,
            'num_folds': len(model_df),
            'map_mean': model_df['map'].mean(),
            'map_std': model_df['map'].std(),
            'map_50_mean': model_df['map_50'].mean(),
            'map_50_std': model_df['map_50'].std(),
            'map_75_mean': model_df['map_75'].mean(),
            'map_75_std': model_df['map_75'].std(),
            'mar_100_mean': model_df['mar_100'].mean(),
            'mar_100_std': model_df['mar_100'].std(),
        }
        
        # Per-class averages
        for class_name in CLASS_NAMES:
            if f'map_{class_name}' in model_df.columns:
                summary[f'map_{class_name}_mean'] = model_df[f'map_{class_name}'].mean()
        
        summary_results.append(summary)

summary_df = pd.DataFrame(summary_results)
summary_file = 'model_evaluation_summary.csv'
summary_df.to_csv(summary_file, index=False)
print(f"✓ Summary results saved to: {summary_file}")

# Print summary table
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}\n")
print(summary_df.to_string(index=False))

print(f"\n{'='*80}")
print("EVALUATION COMPLETE")
print(f"{'='*80}")